In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
df=pd.read_csv("tox21.csv")

In [5]:
df.head()

,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53,mol_id,smiles
0,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,0.0,0.0,TOX3021,CCOc1ccc2nc(S(N)(=O)=O)sc2c1
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3020,CCN1C(=O)NC(c2ccccc2)C1=O
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,0.0,NaN,NaN,TOX3024,CC[C@]1(O)CC[C@H]2[C@@H]3CCC4=CCCC[C@@H]4[C@H]...
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,NaN,0.0,0.0,TOX3027,CCCN(CC)C(CC)C(=O)Nc1c(C)cccc1C
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TOX20800,CC(O)(P(=O)(O)O)P(=O)(O)O


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7831 entries, 0 to 7830
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   NR-AR          7265 non-null   float64
 1   NR-AR-LBD      6758 non-null   float64
 2   NR-AhR         6549 non-null   float64
 3   NR-Aromatase   5821 non-null   float64
 4   NR-ER          6193 non-null   float64
 5   NR-ER-LBD      6955 non-null   float64
 6   NR-PPAR-gamma  6450 non-null   float64
 7   SR-ARE         5832 non-null   float64
 8   SR-ATAD5       7072 non-null   float64
 9   SR-HSE         6467 non-null   float64
 10  SR-MMP         5810 non-null   float64
 11  SR-p53         6774 non-null   float64
 12  mol_id         7831 non-null   object 
 13  smiles         7831 non-null   object 
dtypes: float64(12), object(2)
memory usage: 856.6+ KB


In [7]:
df.isnull().sum()

NR-AR             566
NR-AR-LBD        1073
NR-AhR           1282
NR-Aromatase     2010
NR-ER            1638
NR-ER-LBD         876
NR-PPAR-gamma    1381
SR-ARE           1999
SR-ATAD5          759
SR-HSE           1364
SR-MMP           2021
SR-p53           1057
mol_id              0
smiles              0
dtype: int64

In [8]:
target = "NR-AR"   

In [9]:
df_clean = df[['smiles', target]].dropna()

In [10]:
df_clean[target] = df_clean[target].astype(int)

In [11]:
df_clean[target].value_counts()

NR-AR
0    6956
1     309
Name: count, dtype: int64

In [12]:
df_clean = df[['smiles', 'NR-AR']].dropna()
df_clean['NR-AR'] = df_clean['NR-AR'].astype(int)

In [13]:
df_clean['smiles_len'] = df_clean['smiles'].str.len()
df_clean['num_C'] = df_clean['smiles'].str.count('C')

X = df_clean[['smiles_len', 'num_C']]
y = df_clean['NR-AR']

In [14]:
df_clean.shape

(7265, 4)

In [15]:
df_clean.isnull().sum().sum()

np.int64(0)

In [16]:
X = df_clean.drop("NR-AR", axis=1)
y = df_clean["NR-AR"]

In [17]:
y.value_counts(normalize=True)

NR-AR
0    0.957467
1    0.042533
Name: proportion, dtype: float64

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,        
    random_state=42
)


In [19]:
X.dtypes

smiles        object
smiles_len     int64
num_C          int64
dtype: object

In [20]:
X = df_clean.drop(columns=["NR-AR", "smiles"])
y = df_clean["NR-AR"]

In [21]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [23]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=400,
    max_depth=15,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_scaled, y_train)

,n_estimators,400
,criterion,'gini'
,max_depth,15
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [24]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = rf.predict(X_test_scaled)
y_prob = rf.predict_proba(X_test_scaled)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.97      0.93      0.95      1391
           1       0.17      0.34      0.23        62

    accuracy                           0.90      1453
   macro avg       0.57      0.63      0.59      1453
weighted avg       0.94      0.90      0.92      1453

ROC-AUC: 0.734502910414879


In [25]:
y_custom = (y_prob >= 0.25).astype(int)

from sklearn.metrics import classification_report
print(classification_report(y_test, y_custom))

              precision    recall  f1-score   support

           0       0.97      0.83      0.90      1391
           1       0.11      0.48      0.18        62

    accuracy                           0.82      1453
   macro avg       0.54      0.66      0.54      1453
weighted avg       0.94      0.82      0.87      1453



In [26]:
import joblib

joblib.dump(rf, "tox_model.pkl")
joblib.dump(scaler, "tox_scaler.pkl")
joblib.dump(X.columns.tolist(), "feature_names.pkl")

['feature_names.pkl']

In [27]:
import joblib

joblib.dump(rf, "tox_model.pkl")
joblib.dump(scaler, "tox_scaler.pkl")
joblib.dump(X.columns.tolist(), "feature_names.pkl")

['feature_names.pkl']

In [29]:
import joblib

joblib.dump(rf, "tox_model.pkl")

['tox_model.pkl']

In [30]:
import joblib

joblib.dump(rf, "tox_scaler.pkl")

['tox_scaler.pkl']

In [31]:
import joblib

joblib.dump(rf, "tox_model.pkl")
joblib.dump(scaler, "tox_scaler.pkl")
joblib.dump(X.columns.tolist(), "feature_names.pkl")

['feature_names.pkl']